# 02 — Coleta de internações hospitalares (DATASUS SIH/SUS)

**Objetivo desta etapa:** obter, para os municípios de SP, o número de
internações de idosos (60+) por capítulos da CID-10 associados à hipótese do
estudo, no período Jan/2022-Jul/2026.

## ⚠️ Correção importante: "por local de residência", não "por local de internação"

A primeira versão desta coleta usava os arquivos do TabNet **por local de
internação** — ou seja, contava a internação no município onde fica o
*hospital*, não onde o *paciente mora*. Isso invalidava o cruzamento
geográfico do estudo: municípios-polo em saúde apareciam com taxas enormes
por atenderem a região inteira, e municípios sem hospital apareciam com
zero.

Os arquivos atuais são **por local de residência**, que é o correto para a
hipótese (queremos a internação do idoso que mora sozinho *naquele*
município). O efeito da troca, medido nos dados:

| | por local de internação | por local de residência |
|---|---|---|
| Municípios com ao menos 1 internação | ~313 de 645 | **645 de 645** |
| Total de internações no estado | ~487 mil | ~488 mil (praticamente igual) |
| Pariquera-Açu | 2.088 | 300 (estava **7x** inflado) |
| Poá e Peruíbe | 0 | 797 e 781 |

O total do estado quase não muda (é a mesma população de internações), mas a
**distribuição entre municípios muda completamente** — que é exatamente o
que o estudo mede. Vale reportar essa correção na Metodologia do artigo.

## Os três arquivos

Um por capítulo da CID-10 (é o nível de filtro que o TabNet oferece — não dá
pra pedir só W00-W19 ou só S72 direto na interface):

| Arquivo | Capítulo CID-10 | O que inclui |
|---|---|---|
| `sih_residencia_lesoes_sp.csv` | XIX — Lesões e causas externas | Quedas (W00-W19), fratura de fêmur (S72), e outras lesões/intoxicações |
| `sih_residencia_sintomas_sp.csv` | XVIII — Sintomas e sinais mal definidos | Síncope (R55), confusão mental (R41), e outros sintomas mal definidos |
| `sih_residencia_tmentais_sp.csv` | V — Transtornos mentais e comportamentais | Delirium (F05), e também outros transtornos mentais sem relação com isolamento |

⚠️ **Limitação a declarar no artigo:** cada capítulo é mais largo do que o
subgrupo de interesse original. O capítulo XVIII em especial é usado na
literatura de saúde pública como proxy de diagnóstico tardio/impreciso — o
que reforça a hipótese em vez de enfraquecê-la. Já o capítulo V é o mais
largo dos três (inclui transtornos por uso de substâncias, esquizofrenia
etc.) — vale tratar como complementar/exploratório, não como pilar central
do argumento.

⚠️ **Sem quebra por ano:** estes exports trazem o **total acumulado** do
período (uma única coluna "Internações"), não uma coluna por ano. Por isso o
estudo trabalha com totais por município, e não há análise de série
temporal. Para recuperá-la, seria preciso re-exportar do TabNet com
**Coluna = "Ano processamento"**.

**Saída desta etapa:** `data/processed/internacoes_sp.csv`, com uma linha por
(município, causa) e a contagem de internações no período.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import io


## 2.1 Parser do formato TabNet (testado com os arquivos reais)

O TabNet exporta em **Latin-1** (não UTF-8), separado por `;`, com linhas de
metadados antes da tabela e notas de rodapé depois — por isso não dá pra usar
`pd.read_csv` direto no arquivo, é preciso recortar só o miolo primeiro.

A função foi validada contra os totais impressos no rodapé de cada CSV (batem
exatamente), e o cruzamento por nome foi validado contra o Censo: os 645
municípios batem, com o único ajuste de grafia ("São Luís" vs. "São Luiz" do
Paraitinga) já tratado em `config.ALIASES_MUNICIPIO`.


In [ ]:
def parse_tabnet_sih(path, causa_label):
    """Lê um export do TabNet SIH por local de residência (uma causa/capítulo,
    todos os municípios de SP, total do período) e devolve um DataFrame longo:
    municipio_norm, causa, internacoes."""
    with open(path, encoding="latin1") as f:
        lines = f.readlines()

    # a tabela começa no cabeçalho "Município" e termina antes da linha
    # "Total" (soma geral) seguida das notas de rodapé
    header_idx = next(i for i, l in enumerate(lines) if l.startswith('"Munic'))
    end_idx = next(i for i, l in enumerate(lines) if l.startswith('"Total"'))

    df = pd.read_csv(io.StringIO("".join(lines[header_idx:end_idx])), sep=";", quotechar='"')
    df.columns = ["municipio_raw", "internacoes"]

    # a primeira coluna vem como "350010 ADAMANTINA" -- descarta o código e
    # normaliza o nome, que é a chave de cruzamento entre as bases
    nome = df["municipio_raw"].str.replace(r"^\d{6}\s+", "", regex=True)
    df["municipio_norm"] = nome.apply(config.normalizar_municipio)
    df["internacoes"] = df["internacoes"].replace("-", "0").astype(int)  # "-" no TabNet = zero
    df["causa"] = causa_label

    return df[["municipio_norm", "causa", "internacoes"]]


In [ ]:
arquivos = {
    "lesoes_causas_externas": "sih_residencia_lesoes_sp.csv",
    "sintomas_sinais_maldefinidos": "sih_residencia_sintomas_sp.csv",
    "transtornos_mentais": "sih_residencia_tmentais_sp.csv",
}

internacoes = pd.concat(
    [parse_tabnet_sih(config.DATA_EXTERNAL / arq, causa) for causa, arq in arquivos.items()],
    ignore_index=True,
)

print(internacoes.shape)
print(f"{internacoes['municipio_norm'].nunique()} municípios distintos")
print(internacoes.groupby("causa")["internacoes"].sum())
internacoes.head()


**Conferência:** os totais acima devem bater com a linha "Total" do rodapé de
cada CSV: 350.018 (lesões), 108.899 (sintomas), 29.100 (transtornos mentais).

Note que transtornos mentais aparece em 584 municípios, não 645 — nesse
capítulo, 61 municípios não tiveram nenhuma internação de idoso no período.
O notebook 03 trata isso preenchendo com 0.


## 2.2 Salvar resultado consolidado


In [ ]:
internacoes.to_csv(config.DATA_PROCESSED / "internacoes_sp.csv", index=False)
print("Salvo em", config.DATA_PROCESSED / "internacoes_sp.csv")


## 2.3 (Opcional / mais adiante) Microdados via `pysus`

Se um dia quisermos refinar para o nível de subcategoria (só W00-W19, só
S72 etc., em vez do capítulo inteiro), o caminho é baixar os microdados de
AIH direto do SIH e classificar `DIAG_PRINC` nós mesmas — `config.py` já
tem `CAUSAS_CID10` e `classificar_causa()` prontos para isso. **Não é
necessário agora** (a seção 2.1 já resolve a coleta principal) — deixamos
aqui só como próximo passo possível para aumentar a precisão depois.

⚠️ **Testado neste ambiente de desenvolvimento (sem acesso à internet
externa):** `pip install pysus` funciona (o PyPI é acessível), mas a
chamada que baixa os dados de fato (`pysus.sih(...)`) falha com
`ProxyError 403 Forbidden`, porque o proxy daqui não libera acesso aos
servidores do DATASUS. Ou seja: **esta célula só vai funcionar rodando
localmente**, no seu computador, fora deste ambiente. Note também que a
API do `pysus` mudou de versão: a função antiga
`pysus.online_data.SIH.download` não existe mais nas versões recentes — a
célula abaixo já usa a API atual (`pysus.sih(...)`).


In [ ]:
try:
    import pysus
    PYSUS_OK = True
except ImportError as e:
    print("pysus não instalado (rode: pip install pysus). Não é necessário para o caminho principal:", e)
    PYSUS_OK = False

# Exemplo, para rodar localmente (fora deste ambiente, que bloqueia o acesso ao DATASUS):
# df_mes = pysus.sih(config.UF_SIGLA, 2022, 1, as_dataframe=True)
# df_mes["causa"] = df_mes["DIAG_PRINC"].apply(config.classificar_causa)
